# speech2face on RunPod — PersonaPlex 7B + IMTalker real-time talking avatar

Full-duplex speech-to-speech (NVIDIA PersonaPlex 7B) driving a lip-synced talking head in the
browser. You talk, it answers in the Robert_5 voice while the face moves, and you can talk over it.

This notebook runs the **same configuration as the Jetson AGX Orin build** — int4 trunk,
Robert_5 one-pass depformer, GAN-finetuned Mimi decoder with a TensorRT SEANet, Mimi encoder
CUDA graph, causal motion student + distilled renderer — on **any x86_64 NVIDIA GPU**
(5090, 4090, A100, H100, L40S, A6000 …).

### Before you start

| | |
|---|---|
| **GPU** | compute capability ≥ 8.0 (Ampere or newer). ~14 GB VRAM with the int4 trunk; ~26 GB if this card cannot run ATen's int4 kernels (the notebook detects that and tells you) |
| **Disk** | **≥ 60 GB** on `/workspace` (weights ~19 GB, Python env ~8 GB, HF cache) |
| **Pod template** | a RunPod *PyTorch* or *CUDA 12* image is fine — the notebook builds its own Python 3.10 env, so the base image's Python does not matter |
| **Exposed port** | add **8991** to the pod's *HTTP ports* if you want the RunPod proxy URL. Otherwise use the Cloudflare tunnel cell — it needs no pod configuration |
| **HF token** | a Hugging Face token whose account has **accepted the licence of [`nvidia/personaplex-7b-v1`](https://huggingface.co/nvidia/personaplex-7b-v1)** (gated — accept it on the model page first, or the weight download fails) |

### What the cells do

1. check the GPU, driver and disk
2. install the handful of system libraries the server needs
3. clone the repo into `/workspace/speech2face`
4. write the x86_64 build scripts into the clone
5. `setup_env_x86.sh` — Python 3.10 env in `./env` (~8–20 min)
6. `prepare_x86.sh` — ~19 GB of weights, GPU capability probe, TensorRT engine (~15–40 min)
7. start the server, wait for it to load, print an **HTTPS** URL (the microphone needs HTTPS)
8. headless end-to-end check with `tools/test_session.py`

Total first run: roughly 40–70 minutes, almost all of it downloading. Everything lands on
`/workspace`, so a restarted pod with the same volume skips steps 5 and 6.


## 1 · Configuration

The only thing you normally change here is the Hugging Face token.


In [ ]:
import os, subprocess, sys, time, json, shutil, urllib.request, getpass

# ----------------------------------------------------------------- settings
REPO_URL  = "https://github.com/MoshiHead/speech2face_orin_n_original.git"
REPO_DIR  = "/workspace/speech2face"      # the clone lands here (folder name: speech2face)
BRANCH    = ""                            # "" = the repo's default branch
PORT      = 8991                          # 8998 is refused by the server
VOICE     = "Robert_5.pt"                 # the depformer and decoder are trained for Robert_5
TEXT_PROMPT = ("You are Robert, an assistant from RB Labs. You have knowledge about crypto, "
               "finance, investment, technology, and general topics. Answer every question "
               "completely, descriptively, and with useful detail.")

# Hugging Face token, in order of preference:
#   1. the HF_TOKEN environment variable (set it in the pod's env vars - best on a shared pod)
#   2. /workspace/hf_token.txt
#   3. a prompt, below
# Pasting it straight into this cell also works, but then do not commit the notebook.
token = "GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
HF_TOKEN = "hf_" + token
if not HF_TOKEN and os.path.exists("/workspace/hf_token.txt"):
    HF_TOKEN = open("/workspace/hf_token.txt").read().strip()
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_"), "that does not look like a Hugging Face token"
os.environ["HF_TOKEN"] = HF_TOKEN

# ----------------------------------------------------------------- helper
def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (REPO_DIR if os.path.isdir(REPO_DIR) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

print("repo   :", REPO_URL)
print("into   :", REPO_DIR)
print("port   :", PORT)
print("token  : %s...%s  (%d chars)" % (HF_TOKEN[:6], HF_TOKEN[-4:], len(HF_TOKEN)))


## 2 · Check the GPU, driver and disk

A compute capability below 8.0, a driver older than 525, or less than ~60 GB free on `/workspace` will stop you later — better to see it now.


In [ ]:
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/")
sh("echo; df -h /workspace / 2>/dev/null | sed -n '1,5p'; echo; uname -m", cwd="/")

# a quick sanity read of the numbers above
out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB of VRAM" % (cap, gib))
    if float(cap) < 8.0:
        print("!! this pipeline needs Ampere or newer (compute capability >= 8.0)")
    elif gib < 20:
        print("!! under 20 GiB of VRAM - expect CUDA OOM even with the int4 trunk")
    else:
        print("ok")


## 3 · System libraries

`libsndfile1` for soundfile, `ffmpeg` for torchaudio/librosa, `libgl1` + `libglib2.0-0` for OpenCV, `util-linux` for the watchdog's `flock`. Already present in most RunPod images, in which case this is a no-op.


In [ ]:
sh("export DEBIAN_FRONTEND=noninteractive && "
   "apt-get update -qq && "
   "apt-get install -y -qq --no-install-recommends "
   "  git curl ca-certificates bzip2 unzip util-linux "
   "  ffmpeg libsndfile1 libgl1 libglib2.0-0 "
   "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/")


## 4 · Clone the repo

Re-running this cell on an existing clone just fetches and fast-forwards it; the weights you already downloaded into `weights/`, `assets/` and `hf/` are untouched.


In [ ]:
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    # FETCH_HEAD, not origin/HEAD: a --depth 1 clone has no remote HEAD ref
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

# every later cell (including the %%writefile ones) is relative to the clone
os.chdir(REPO_DIR)
print("\ncwd:", os.getcwd())
sh("ls -la", check=False)


## 5 · Write the x86_64 build scripts into the clone

The repo's `setup_env.sh` / `prepare.sh` / `run.sh` are the **Jetson** scripts: they refuse to run
on anything but `aarch64`, install torch from NVIDIA's Jetson wheel index, take TensorRT from
JetPack, and load the bundled **aarch64** `sphn` from `src/pylibs`. The cells below write the
x86_64 counterparts next to them, so the Jetson path keeps working untouched:

| file | what it changes |
|---|---|
| `requirements-x86-torch.txt` | torch 2.8.0 / torchaudio 2.8.0 / torchvision 0.23.0 / triton 3.4.0, **CUDA 12.8** builds (covers Ampere, Ada and Blackwell `sm_120`) |
| `requirements-x86-extra.txt` | `sphn==0.2.1` and `tensorrt-cu12` from PyPI, which on the Orin come from `src/pylibs` and JetPack |
| `setup_env_x86.sh` | Miniforge **x86_64**, Python 3.10 env in `./env` |
| `prepare_x86.sh` | the same ~19 GB of weights; builds the TensorRT engine with the Python API because the pip TensorRT wheels ship no `trtexec`; a failed engine build is a warning, not an error |
| `run_x86.sh` | keeps `src/pylibs` **off** `PYTHONPATH`, makes the TensorRT engine optional, and takes the int4 setting from the GPU probe |
| `run_watchdog_x86.sh` | restarts `run_x86.sh` if it dies |
| `tools/build_trt_engine.py` | ONNX → TensorRT engine via the Python API (the `trtexec` replacement) |
| `tools/gpu_probe.py` | runs ATen's int4 tinygemm kernels on *this* GPU and writes `.gpu_probe.env` |

If the repo already contains these files, the cells simply overwrite them with the same content.


In [ ]:
%%writefile requirements-x86-torch.txt
# x86_64 (RunPod / any desktop NVIDIA GPU) torch stack -- CUDA 12.8 builds.
# cu128 covers Ampere (A100/A6000), Ada (4090/L40S) and Blackwell (5090/B200, sm_120).
# Same versions as the Jetson build so the rest of requirements.txt matches exactly.
#     pip install -r requirements-x86-torch.txt
# The +cu128 local version makes the pin unambiguous even with PyPI as an extra index.
--index-url https://download.pytorch.org/whl/cu128
--extra-index-url https://pypi.org/simple
torch==2.8.0+cu128
torchaudio==2.8.0+cu128
torchvision==0.23.0+cu128
triton==3.4.0


In [ ]:
%%writefile requirements-x86-extra.txt
# x86_64 extras that on a Jetson come from JetPack or the bundled wheel:
#   sphn      -> bundled aarch64 .so in src/pylibs on the Orin; PyPI has an x86_64 wheel
#   tensorrt  -> JetPack on the Orin; pip wheel (bundles its own CUDA 12 libs) here
# Keep sphn at 0.2.x: the server needs OpusStreamReader/Writer.
sphn==0.2.1
tensorrt-cu12==10.13.3.9


In [ ]:
%%writefile setup_env_x86.sh
#!/usr/bin/env bash
# setup_env_x86.sh -- create the Python environment for this package on an x86_64 NVIDIA GPU box
# (RunPod, Lambda, a local workstation). The Jetson counterpart is ./setup_env.sh.
#
#   ./setup_env_x86.sh
#
# Installs Miniforge (conda-forge, free) into ./.miniforge, creates a Python 3.10 env in ./env,
# installs torch 2.8.0 / torchaudio 2.8.0 / torchvision 0.23.0 / triton 3.4.0 built for CUDA 12.8
# (requirements-x86-torch.txt), then requirements.txt, then sphn + TensorRT from pip
# (requirements-x86-extra.txt). Nothing is installed outside this folder (no apt, no sudo).
# Takes ~8-20 min (downloads ~5 GB). prepare_x86.sh and run_x86.sh use ./env automatically.
# Safe to re-run: finished steps are skipped.
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
MF="$HERE/.miniforge"; ENV="$HERE/env"
MF_URL="${MF_URL:-https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh}"
# keep every cache next to the package so it lands on the persistent volume, not the container disk
export CONDA_PKGS_DIRS="$HERE/.cache/conda-pkgs" PIP_CACHE_DIR="$HERE/.cache/pip" TMPDIR="$HERE/.cache/tmp"
mkdir -p "$CONDA_PKGS_DIRS" "$PIP_CACHE_DIR" "$TMPDIR"
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/5  checks"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./setup_env.sh"
command -v nvidia-smi >/dev/null || die "nvidia-smi not found -- no NVIDIA driver visible in this container"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /'
[ -n "$(nvidia-smi --query-gpu=name --format=csv,noheader)" ] || die "no GPU visible"

log "2/5  Miniforge -> $MF"
if [ ! -x "$MF/bin/conda" ]; then
  curl -fL --retry 3 -o "$TMPDIR/miniforge.sh" "$MF_URL" || die "could not download Miniforge from $MF_URL"
  bash "$TMPDIR/miniforge.sh" -b -p "$MF" > "$TMPDIR/miniforge_install.log" 2>&1 \
    || { tail -20 "$TMPDIR/miniforge_install.log"; die "Miniforge install failed"; }
  rm -f "$TMPDIR/miniforge.sh"
fi
"$MF/bin/conda" --version | sed 's/^/    /'

log "3/5  Python 3.10 env -> $ENV (3.10: tokenizers 0.13.3 / transformers 4.30.2 have no newer wheels)"
if [ ! -x "$ENV/bin/python" ]; then
  "$MF/bin/conda" create -y -q -p "$ENV" python=3.10 pip > "$TMPDIR/conda_create.log" 2>&1 \
    || { tail -20 "$TMPDIR/conda_create.log"; die "conda create failed"; }
fi
PY="$ENV/bin/python"
"$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10), sys.version' || die "env python is not 3.10"
"$PY" -m pip install -q --upgrade pip

log "4/5  torch/torchaudio/torchvision/triton for CUDA 12.8, then requirements.txt (~5 GB)"
if ! "$PY" -c 'import torch, sys; sys.exit(0 if torch.version.cuda and torch.cuda.is_available() else 1)' 2>/dev/null; then
  "$PY" -m pip install -r "$HERE/requirements-x86-torch.txt" \
    || die "torch install failed (index: https://download.pytorch.org/whl/cu128)"
fi
"$PY" -c 'import torch, sys; print("    torch", torch.__version__, "CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0)); sys.exit(0 if torch.cuda.is_available() else 1)' \
  || die "torch cannot see the GPU"
"$PY" -m pip install -r "$HERE/requirements.txt" || die "pip install -r requirements.txt failed"

log "5/5  sphn + TensorRT from pip (on a Jetson these come from src/pylibs and JetPack)"
"$PY" -m pip install -r "$HERE/requirements-x86-extra.txt" || die "sphn/TensorRT install failed"
# uvloop (pulled in by uvicorn[standard]) made every 80 ms step ~13 ms slower on the reference box
"$PY" -m pip uninstall -y -q uvloop 2>/dev/null || true
"$PY" - <<'T' || die "installed, but the environment check failed"
import importlib.util, torch, triton, torchaudio, torchvision, sphn, fastapi, timm, transformers, cv2
import tensorrt
assert importlib.util.find_spec("uvloop") is None, "uvloop installed: remove it (pip uninstall uvloop)"
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
cc = torch.cuda.get_device_capability(0)
print(f"    torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | "
      f"TensorRT {tensorrt.__version__} | {torch.cuda.get_device_name(0)} sm_{cc[0]}{cc[1]} "
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GiB")
T
rm -rf "$TMPDIR"
log "ENV READY: $ENV   --  next: ./prepare_x86.sh hf_YOUR_TOKEN"


In [ ]:
%%writefile prepare_x86.sh
#!/usr/bin/env bash
# prepare_x86.sh -- weights + TensorRT engine for the PersonaPlex + IMTalker avatar on an
# x86_64 NVIDIA GPU (RunPod, Lambda, a local workstation). The Jetson counterpart is ./prepare.sh.
#
#   ./setup_env_x86.sh                             (once: Python env from scratch in ./env)
#   ./prepare_x86.sh hf_yourtokenhere              (or `hf auth login` first and omit the token)
#   PY=/path/to/python ./prepare_x86.sh hf_...     (to use your own Python 3.10 instead of ./env)
#
# Checks the environment, downloads ~19 GB of weights, probes which fast paths this GPU supports
# and builds the audio-decoder TensorRT engine for THIS GPU. Installs nothing. Then: ./run_x86.sh
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
TOKEN="${1:-${HF_TOKEN:-}}"
[ -n "$TOKEN" ] && export HF_TOKEN="$TOKEN" HUGGING_FACE_HUB_TOKEN="$TOKEN"
export HF_HOME="${HF_HOME:-$HERE/hf}"
# Python: $PY if given, else ./env from setup_env_x86.sh, else python3 on PATH
[ -z "${PY:-}" ] && [ -x "$HERE/env/bin/python" ] && PY="$HERE/env/bin/python"
PY="$(command -v "${PY:-python3}")" || { echo "python3 not found (run ./setup_env_x86.sh, or set PY=...)"; exit 1; }
W="$HERE/weights"
BASE_REPO=nvidia/personaplex-7b-v1
BASE_REV=fdaf4090a61cb315c138a1faee287ffd6c716309
OURS_REPO=niloy629/personaplex-parallel-depformer
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
warn(){ printf '\033[33mWARNING: %s\033[0m\n' "$*" >&2; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/6  environment check ($PY)"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64; on a Jetson use ./prepare.sh"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /' || die "nvidia-smi failed"
"$PY" - <<'T' || die "environment incomplete -- run ./setup_env_x86.sh (see README_RUNPOD.md)"
import sys
assert sys.version_info[:2] == (3, 10), f"need Python 3.10, got {sys.version.split()[0]}"
import torch, torchaudio, torchvision, triton, sphn, fastapi, uvicorn, timm, transformers, sentencepiece, cv2, librosa, torchdiffeq, huggingface_hub
assert torch.cuda.is_available(), "torch has no CUDA"
assert hasattr(sphn, "OpusStreamReader"), "sphn too old -- need 0.2.x"
f = torch.compile(lambda x: torch.sin(x) * 2 + 1)
x = torch.randn(1024, device="cuda"); assert torch.allclose(f(x), torch.sin(x) * 2 + 1, atol=1e-5)
print(f"    python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | triton {triton.__version__} | torch.compile OK")
T
echo "$PY" > "$HERE/.python_path"

log "2/6  base model from $BASE_REPO (~16 GB, gated: needs your token + accepted licence)"
"$PY" - <<PYEOF || die "base download failed: is the token valid and the model licence accepted on HF?"
import os
from huggingface_hub import snapshot_download
p = snapshot_download("$BASE_REPO", revision="$BASE_REV", token=os.environ.get("HF_TOKEN"),
                      allow_patterns=["model.safetensors", "tokenizer-e351c8d8-checkpoint125.safetensors",
                                      "tokenizer_spm_32k_3.model", "voices.tgz"])
open(os.path.join("$HERE", ".snap_path"), "w").write(p); print("    ->", p)
PYEOF
SNAP="$(cat "$HERE/.snap_path")"
mkdir -p "$W"
ln -sfn "$SNAP/tokenizer-e351c8d8-checkpoint125.safetensors" "$W/mimi.safetensors"
ln -sfn "$SNAP/tokenizer_spm_32k_3.model" "$W/tokenizer_spm_32k_3.model"
[ -d "$W/voices" ] || tar xzf "$SNAP/voices.tgz" -C "$W"
[ -s "$W/voices/NATM0.pt" ] || die "voices.tgz did not contain voices/NATM0.pt"

fetch() {  # fetch <hf folder> <dest dir> <files...> : download from our public repo, sha256-check, link
  "$PY" - "$@" <<PYEOF || die "download from $OURS_REPO/\$1 failed"
import os, sys, hashlib
from huggingface_hub import hf_hub_download
folder, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]
os.makedirs(dest, exist_ok=True)
want = {}
for line in open(hf_hub_download("$OURS_REPO", f"{folder}/SHA256SUMS")):
    h, f = line.split(); want[os.path.basename(f)] = h
for f in files:
    src = hf_hub_download("$OURS_REPO", f"{folder}/{f}")
    h = hashlib.sha256()
    with open(src, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want[f]:
        raise SystemExit(f"hash mismatch for {f}")
    dst = os.path.join(dest, f)
    if os.path.islink(dst) or os.path.exists(dst): os.remove(dst)
    os.symlink(src, dst)
print(f"    {folder}: {len(files)} files, sha256 OK")
PYEOF
}

log "3/6  Robert_5 one-pass depformer + GAN audio decoder from $OURS_REPO (~2.9 GB, public)"
fetch robert5_gan "$W" pd_quant_onpol_robert5_plaus2.pt mimi_dec_gan_robert5.pt

log "4/6  avatar weights + Robert_5 voice + decoder ONNX from $OURS_REPO (~250 MB, public)"
fetch avatar_assets "$HERE/assets" motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt \
  blink_motion.pt ref.jpeg Robert_5.pt seanet_w12.onnx
for f in motion_student_38M.pt renderer_narrow.pt silence_helium_mean.pt blink_motion.pt ref.jpeg; do
  ln -sfn "$HERE/assets/$f" "$W/$f"
done
ln -sfn "$HERE/assets/Robert_5.pt" "$W/voices/Robert_5.pt"

log "5/6  GPU capability probe (int4 trunk kernels, VRAM, TensorRT)"
"$PY" "$HERE/tools/gpu_probe.py" "$HERE/.gpu_probe.env" || die "GPU probe failed"

log "6/6  TensorRT engine for the audio decoder (built for THIS GPU, ~1-5 min)"
# Optional: if it cannot be built the server keeps Mimi's PyTorch SEANet decoder (a few ms
# slower per step, identical audio), so a failure here is a warning, not an error.
TRTEXEC="${TRTEXEC:-/usr/src/tensorrt/bin/trtexec}"
if [ ! -s "$W/seanet_w12_fp32.engine" ]; then
  if [ -x "$TRTEXEC" ]; then
    "$TRTEXEC" --onnx="$HERE/assets/seanet_w12.onnx" --saveEngine="$W/seanet_w12_fp32.engine" \
      > "$W/trtexec_build.log" 2>&1 || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  else
    "$PY" "$HERE/tools/build_trt_engine.py" "$HERE/assets/seanet_w12.onnx" \
      "$W/seanet_w12_fp32.engine" > "$W/trtexec_build.log" 2>&1 \
      || { tail -20 "$W/trtexec_build.log"; rm -f "$W/seanet_w12_fp32.engine"; }
  fi
fi
if [ -s "$W/seanet_w12_fp32.engine" ]; then
  "$PY" - <<PYEOF || { warn "engine does not load -- removing it, the PyTorch decoder will be used"; rm -f "$W/seanet_w12_fp32.engine"; }
import tensorrt as trt
e = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(open("$W/seanet_w12_fp32.engine", "rb").read())
assert e is not None
print("    engine OK:", [(e.get_tensor_name(i), tuple(e.get_tensor_shape(e.get_tensor_name(i)))) for i in range(e.num_io_tensors)])
PYEOF
else
  warn "no TensorRT engine (log: weights/trtexec_build.log) -- the server will use Mimi's PyTorch SEANet decoder"
fi
log "READY  --  ./run_x86.sh   then open the HTTPS URL for port \${PORT:-8991}"


In [ ]:
%%writefile run_x86.sh
#!/usr/bin/env bash
# run_x86.sh -- PersonaPlex + IMTalker real-time avatar on an x86_64 NVIDIA GPU
# (RunPod / Lambda / a local workstation). The Jetson counterpart is ./run.sh and keeps the
# tuned Orin settings; this script runs the same configuration with the two Jetson-only
# assumptions removed:
#   * sphn comes from pip, not from the bundled aarch64 .so in src/pylibs
#   * the TensorRT SEANet engine is optional (the PyTorch decoder is used if it is missing)
# and with int4 trunk quantisation driven by what tools/gpu_probe.py found this GPU can do.
#
#   ./run_x86.sh                 defaults (port 8991)
#   PORT=8080 ./run_x86.sh       another port (not 8998: refused by the server)
#   TRY19_INT4_G=0 ./run_x86.sh  bf16 trunk instead of int4 (needs ~26 GB of VRAM)
set -u
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
SRC="$HERE/src"; W="$HERE/weights"
[ -s "$HERE/.snap_path" ] && [ -s "$HERE/.python_path" ] || { echo "run ./prepare_x86.sh <hf_token> first"; exit 1; }
PY="${PY:-$(cat "$HERE/.python_path")}"   # the Python prepare_x86.sh checked
SNAP="$(cat "$HERE/.snap_path")"
PORT="${PORT:-8991}"   # 8998 is refused by the server (reserved for a legacy production server)
export HF_HOME="${HF_HOME:-$HERE/hf}"
# compiled-kernel caches next to the package so they survive on the persistent volume
export TRITON_CACHE_DIR="${TRITON_CACHE_DIR:-$HERE/.cache/triton}" TORCHINDUCTOR_CACHE_DIR="${TORCHINDUCTOR_CACHE_DIR:-$HERE/.cache/inductor}"
export DESIGN_A_DIR="$SRC/niloys" IMTALKER_DIR="$SRC/imtalker"
# NOTE: src/pylibs is deliberately NOT on the path -- the sphn there is an aarch64 build.
export PYTHONPATH="$SRC/niloys:$SRC/moshi_pkg:${PYTHONPATH:-}"
unset TRY19_DEC_CKPT
export NATIVE_PASSTHROUGH=1
export TRY19_STUDENT_GRAPH="${TRY19_STUDENT_GRAPH:-1}" TRY19_RENDER_FP16="${TRY19_RENDER_FP16:-1}" TRY19_PD_REFINE="${TRY19_PD_REFINE:-0}"
export TRY19_PD="$W/pd_quant_onpol_robert5_plaus2.pt"
export TRY19_MIMI_DEC_FT="$W/mimi_dec_gan_robert5.pt"
export TRY19_MIMI_GRAPH="${TRY19_MIMI_GRAPH:-1}" TRY19_MIMI_PARTS="${TRY19_MIMI_PARTS:-seanet_quant}" TRY19_MIMI_CAPTURE_AT_INIT="${TRY19_MIMI_CAPTURE_AT_INIT:-1}"
export TRY19_CODE_GUARD="${TRY19_CODE_GUARD:-1}"

# int4 trunk: ATen's tinygemm kernels are not built for every CUDA architecture, so
# prepare_x86.sh probed them and wrote the verdict. An explicit value in the environment wins.
_int4_user="${TRY19_INT4_G:-}"; _gate_user="${TRY19_DEP_GATING_INT4:-}"
if [ -s "$HERE/.gpu_probe.env" ]; then set -a; . "$HERE/.gpu_probe.env"; set +a; fi
export TRY19_INT4_G="${_int4_user:-${TRY19_INT4_G:-32}}"
export TRY19_DEP_GATING_INT4="${_gate_user:-${TRY19_DEP_GATING_INT4:-1}}"

# TensorRT SEANet decoder: optional here. Without it Mimi's PyTorch SEANet is used.
if [ -s "$W/seanet_w12_fp32.engine" ]; then
  export TRY19_TRT_SEANET="$W/seanet_w12_fp32.engine"
else
  unset TRY19_TRT_SEANET
  echo "note: no weights/seanet_w12_fp32.engine -- using Mimi's PyTorch SEANet decoder"
fi

mkdir -p "$SRC/niloys/live/static" "$HERE/captures"
ln -sfn "$SRC/imtalker/static/assets" "$SRC/niloys/live/static/assets"
ln -sfn "$SRC/niloys/static/index_try21_no_idle_dissolve.html" "$SRC/niloys/live/static/index.html"
echo "x86: trunk int4 G=$TRY19_INT4_G + robert5 one-pass depformer + GAN decoder$([ -n "${TRY19_TRT_SEANET:-}" ] && echo ' (TensorRT)') voice=${VOICE:-Robert_5.pt} port=$PORT"
exec "$PY" -u "$SRC/niloys/live/imtalker_try19_student_live.py" \
  --live_module imtalker_native_passthrough \
  --student_ckpt "$W/motion_student_38M.pt" --distilled_renderer "$W/renderer_narrow.pt" \
  --motion_gain "${GAIN:-1.2}" --dep_q 8 \
  --blend_frames 0 --jump_thresh 0.15 --mouth_close 0.5 \
  --host 0.0.0.0 --port "$PORT" \
  --html_path "$SRC/niloys/static/index_try21_no_idle_dissolve.html" \
  --generator_path "$W/generator_last.ckpt" --renderer_path "$W/renderer.ckpt" \
  --adapter_path "$W/adapter_last.pt" \
  --adapter_type unitalk_last_layer --adapter_num_layers 12 --adapter_dropout 0.0 \
  --adapter_window_mode lookahead --adapter_future_steps 0 \
  --ref_path "$W/ref.jpeg" --wav2vec_model_path "$W/wav2vec2-base-960h" \
  --moshi_root "$W" --mimi_hf_repo nvidia/personaplex-7b-v1 \
  --moshi_weight "$SNAP/model.safetensors" --mimi_weight "$W/mimi.safetensors" \
  --tokenizer "$W/tokenizer_spm_32k_3.model" \
  --text_prompt "${TEXT_PROMPT:-You are Robert, an assistant from RB Labs. You have knowledge about crypto, finance, investment, technology, and general topics. Answer every question completely, descriptively, and with useful detail.}" \
  --voice_prompt "${VOICE:-Robert_5.pt}" --voice_prompt_dir "$W/voices" \
  --enable_moshi_reply --direct_reply_hidden --reply_hidden_steps_per_chunk 1 \
  --audio_chunk_sec 2.0 --wav2vec_sec 2.0 --fm_chunk_frames 2 --helium_deque_size 25 \
  --prebuffer_chunks "${PREBUFFER:-3}" --render_sub_batch 2 --renderer_precision fp32 \
  --frame_q_backpressure 3 --buffer_ms 80 --skip_fm_audio_encoder \
  --assistant_speech_rms_threshold 0.006 --assistant_speech_hold_chunks 1 \
  --motion_ref_blend 0.0 --motion_prior_noise_blend 0.0 \
  --a_cfg_scale 1.24 --nfe 3 --seed 42 --noise_seed 42 --shared_noise \
  --fp32 --tf32 \
  --silence_helium_path "$W/silence_helium_mean.pt" \
  --jpeg_quality 90 --device cuda --reply_audio_gain 1.0 --output_audio_codec opus \
  --capture_dir "$HERE/captures" \
  --blink_motion_path "$W/blink_motion.pt"


In [ ]:
%%writefile run_watchdog_x86.sh
#!/usr/bin/env bash
# run_watchdog_x86.sh -- keep ./run_x86.sh up: restart it if it exits or hits a CUDA device-side assert.
# Log: run.log (current), crashlogs/ (saved logs of crashed runs).   Stop: pkill -f run_watchdog_x86.sh; pkill -f speech2face/src
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"; mkdir -p crashlogs
exec 9>"$HERE/.watchdog_x86.lock"; flock -n 9 || { echo "watchdog already running"; exit 1; }
while true; do
  ./run_x86.sh > run.log 2>&1 &
  PID=$!; echo "[$(date +%F_%T)] started pid $PID" >> crashlogs/watchdog.log
  sleep 60
  while kill -0 $PID 2>/dev/null; do
    if grep -q "device-side assert" run.log; then
      cp run.log crashlogs/crash_$(date +%m%d_%H%M%S).log
      echo "[$(date +%F_%T)] device-side assert -> restarting" >> crashlogs/watchdog.log
      kill -9 $PID; break
    fi
    sleep 5
  done
  wait $PID 2>/dev/null; sleep 5
done


In [ ]:
%%writefile tools/build_trt_engine.py
"""tools/build_trt_engine.py -- build a TensorRT engine from an ONNX file with the Python API.

    python tools/build_trt_engine.py <model.onnx> <out.engine> [workspace_GiB]

Same job as `trtexec --onnx=... --saveEngine=...`, for machines where trtexec is not installed
(the pip TensorRT wheels ship the libraries and the Python bindings but no trtexec binary;
JetPack ships trtexec at /usr/src/tensorrt/bin/trtexec). Engines are tied to the GPU and the
TensorRT version, so this always runs on the machine that will serve.

The SEANet window decoder used here has fully static shapes, so no optimisation profile is
needed; precision stays fp32 to match weights/seanet_w12_fp32.engine on the Orin.
"""
import os
import sys

import tensorrt as trt


def build(onnx_path: str, engine_path: str, workspace_gib: float = 4.0) -> None:
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network()
    parser = trt.OnnxParser(network, logger)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            for i in range(parser.num_errors):
                print(f"onnx parse error: {parser.get_error(i)}", file=sys.stderr)
            raise SystemExit(f"could not parse {onnx_path}")

    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, int(workspace_gib * (1 << 30)))
    print(f"[build_trt_engine] TensorRT {trt.__version__}: {onnx_path} -> {engine_path} "
          f"(fp32, workspace {workspace_gib:g} GiB)", flush=True)
    for i in range(network.num_inputs):
        t = network.get_input(i)
        print(f"    input  {t.name} {tuple(t.shape)} {t.dtype}")
    for i in range(network.num_outputs):
        t = network.get_output(i)
        print(f"    output {t.name} {tuple(t.shape)} {t.dtype}")

    plan = builder.build_serialized_network(network, cfg)
    if plan is None:
        raise SystemExit("TensorRT build_serialized_network returned None")
    tmp = engine_path + ".tmp"
    with open(tmp, "wb") as f:
        f.write(plan)
    os.replace(tmp, engine_path)
    print(f"[build_trt_engine] wrote {engine_path} ({os.path.getsize(engine_path)/2**20:.1f} MiB)",
          flush=True)


if __name__ == "__main__":
    if len(sys.argv) < 3:
        raise SystemExit(__doc__)
    build(sys.argv[1], sys.argv[2], float(sys.argv[3]) if len(sys.argv) > 3 else 4.0)


In [ ]:
%%writefile tools/gpu_probe.py
"""tools/gpu_probe.py -- report the GPU and decide which fast paths this GPU can actually run.

    python tools/gpu_probe.py [out.env]

The Orin configuration quantises the 7B trunk to int4 with ATen's tinygemm kernels
(torch.ops.aten._weight_int4pack_mm). That kernel is not compiled for every CUDA
architecture, so on some cards it raises instead of running. This script tries it with the
exact code path the server uses (src/niloys/int4_linear.py) and writes a shell-sourceable file:

    TRY19_INT4_G=32          # int4 trunk works -> use it (needs ~14 GB of VRAM)
    TRY19_INT4_G=0           # int4 unavailable  -> bf16 trunk (needs ~26 GB of VRAM)
    TRY19_DEP_GATING_INT4=1/0

run_x86.sh sources that file automatically if it is next to the package as .gpu_probe.env.
"""
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, os.path.join(ROOT, "src", "niloys"))

import torch  # noqa: E402

OUT = sys.argv[1] if len(sys.argv) > 1 else os.path.join(ROOT, ".gpu_probe.env")


def main() -> int:
    if not torch.cuda.is_available():
        print("ERROR: torch cannot see a CUDA GPU", file=sys.stderr)
        return 1
    props = torch.cuda.get_device_properties(0)
    cc = (props.major, props.minor)
    vram = props.total_memory / 2 ** 30
    print(f"GPU            : {props.name}  sm_{cc[0]}{cc[1]}  {vram:.1f} GiB")
    print(f"torch          : {torch.__version__} (CUDA {torch.version.cuda})")
    try:
        import tensorrt
        print(f"TensorRT       : {tensorrt.__version__}")
    except Exception as e:
        print(f"TensorRT       : NOT AVAILABLE ({type(e).__name__}) -- the SEANet decoder "
              "stays in PyTorch (a few ms slower, same audio)")

    # bf16 is required: the trunk weights are bf16 and the int4 kernels take bf16 activations.
    if cc[0] < 8:
        print("bf16           : NOT SUPPORTED on sm_%d%d -- this pipeline needs Ampere or newer"
              % cc, file=sys.stderr)
        return 1
    print("bf16           : OK")

    int4_ok, int4_err = False, ""
    try:
        from int4_linear import Int4Linear, group_quantize
        lin = torch.nn.Linear(512, 256, bias=False, device="cuda", dtype=torch.bfloat16)
        packed, sz = group_quantize(lin.weight.data, 32)
        q = Int4Linear(packed, sz, None, 256, 32)
        x = torch.randn(4, 512, device="cuda", dtype=torch.bfloat16)
        ref, got = lin(x).float(), q(x).float()
        cos = torch.nn.functional.cosine_similarity(ref.flatten(), got.flatten(), 0).item()
        if not torch.isfinite(got).all():
            raise RuntimeError("int4 matmul produced non-finite values")
        if cos < 0.9:
            raise RuntimeError(f"int4 matmul cosine {cos:.3f} vs bf16 -- kernel misbehaving")
        int4_ok = True
        print(f"int4 tinygemm  : OK (cosine {cos:.4f} vs bf16)")
    except Exception as e:                                   # noqa: BLE001
        int4_err = f"{type(e).__name__}: {e}"
        print(f"int4 tinygemm  : UNAVAILABLE -- {int4_err}")

    try:
        f = torch.compile(lambda t: torch.sin(t) * 2 + 1)
        t = torch.randn(1024, device="cuda")
        assert torch.allclose(f(t), torch.sin(t) * 2 + 1, atol=1e-5)
        print("torch.compile  : OK")
    except Exception as e:                                   # noqa: BLE001
        print(f"torch.compile  : FAILED ({type(e).__name__}: {e})")

    need = 14 if int4_ok else 26
    print(f"\nplan           : {'int4 G=32 trunk' if int4_ok else 'bf16 trunk (int4 kernels unusable)'}"
          f", needs ~{need} GB of VRAM, this GPU has {vram:.0f} GB")
    if vram < need:
        print(f"WARNING        : only {vram:.0f} GB of VRAM -- expect CUDA OOM. Use a bigger GPU "
              f"(>= {need} GB).", file=sys.stderr)

    with open(OUT, "w") as fh:
        fh.write("# written by tools/gpu_probe.py -- sourced by run_x86.sh\n")
        fh.write(f"TRY19_INT4_G={32 if int4_ok else 0}\n")
        fh.write(f"TRY19_DEP_GATING_INT4={1 if int4_ok else 0}\n")
    print(f"wrote          : {OUT}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
sh("chmod +x setup_env_x86.sh prepare_x86.sh run_x86.sh run_watchdog_x86.sh "
   "setup_env.sh prepare.sh run.sh run_watchdog.sh 2>/dev/null; "
   "ls -l *_x86.sh requirements-x86-*.txt tools/build_trt_engine.py tools/gpu_probe.py")


## 6 · Build the Python environment  *(~8–20 min, ~5 GB)*

Miniforge → `./.miniforge`, a Python 3.10 env → `./env`, then torch (CUDA 12.8), the pinned
`requirements.txt`, and `sphn` + TensorRT from pip. Python 3.10 specifically, because the pinned
`transformers==4.30.2` / `tokenizers==0.13.3` have no wheels for anything newer.

Nothing is installed outside the clone — no `pip install` into the base image. Safe to re-run:
finished steps are skipped. The last line must be `ENV READY`.


In [ ]:
t0 = time.time()
sh("./setup_env_x86.sh")
print("\n--- setup_env_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


## 7 · Download the weights and build the TensorRT engine  *(~15–40 min, ~19 GB)*

1. environment check
2. `nvidia/personaplex-7b-v1` — bf16 trunk, Mimi codec, tokenizer, voices (**gated**, ~16 GB)
3. the Robert_5 one-pass depformer + GAN Mimi decoder (~2.9 GB, public, sha256-checked)
4. avatar weights, the Robert_5 voice and the decoder ONNX (~250 MB, public, sha256-checked)
5. the GPU capability probe
6. the TensorRT engine for the audio decoder, built for **this** GPU

Step 2 fails if the token is wrong or the account has not accepted the model licence.
Safe to re-run; it skips what is already there. The last line must be `READY`.


In [ ]:
t0 = time.time()
sh("./prepare_x86.sh " + HF_TOKEN)
print("\n--- prepare_x86.sh took %.1f min ---" % ((time.time() - t0) / 60))


### What the probe decided

`TRY19_INT4_G=32` means the 7B trunk is quantised to int4 at load — what the Orin ships, ~14 GB
of VRAM. `TRY19_INT4_G=0` means ATen's int4 tinygemm kernels are not usable on this card, so the
trunk stays bf16: correct but slower, and it needs ~26 GB of VRAM.


In [ ]:
sh("cat .gpu_probe.env; echo; "
   "ls -lL weights/seanet_w12_fp32.engine 2>/dev/null || echo 'no TensorRT engine "
   "(the PyTorch SEANet decoder will be used - a few ms slower, same audio)'", check=False)


## 8 · Start the server

It runs detached with `setsid`, so it keeps serving while you use the rest of the notebook and
survives a kernel restart. Output goes to `run.log`.


In [ ]:
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_try19_student_live.py; sleep 2", check=False)
sh("PORT=%d VOICE=%s TEXT_PROMPT=%s nohup setsid ./run_x86.sh > run.log 2>&1 < /dev/null & "
   "sleep 3; echo started:; pgrep -af imtalker_try19_student_live.py | head -3"
   % (PORT, VOICE, json.dumps(TEXT_PROMPT)))


### Wait until it has loaded

Loading the 7B trunk, quantising it, capturing the CUDA graphs and compiling the Triton kernels
takes a few minutes the first time (the kernel cache in `.cache/` makes later starts quicker).
This cell polls `/health` until `loaded: true`.


In [ ]:
def wait_loaded(port, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            with urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5) as r:
                h = json.load(r)
            if h.get("loaded"):
                print("\nloaded after %.0fs: %s" % (time.time() - t0, h))
                return True
        except Exception:
            pass
        if subprocess.run(["pgrep", "-f", "imtalker_try19_student_live.py"],
                          capture_output=True).returncode != 0:
            print("\n!! the server process is gone - last 40 lines of run.log:")
            sh("tail -40 run.log", check=False)
            return False
        tail = subprocess.run(["bash", "-c", "tail -1 run.log"], capture_output=True,
                              text=True, cwd=REPO_DIR).stdout.strip()
        print("  ... %5.0fs  %s" % (time.time() - t0, tail[:110]), flush=True)
        time.sleep(15)
    print("\n!! timed out")
    sh("tail -40 run.log", check=False)
    return False

ok = wait_loaded(PORT)
print()
# the feature lines that must be in the log (int4 / trt_seanet are absent if disabled above)
sh("grep -aE 'x86:|int4 G=|GATING|TRY19_PD hook|fine-tune hook|captured AT INIT|trt_seanet|"
   "Uvicorn running' run.log || echo '(none found yet)'", check=False)


## 9 · Open it in Chrome

**The microphone only works on `https://` or `http://localhost`.** Plain `http://<pod-ip>:8991`
loads the page, but Chrome refuses the microphone on it. Two ways to get HTTPS:

### Option A — the RunPod HTTP proxy  *(use this if 8991 is in the pod's HTTP ports)*


In [ ]:
pod = os.environ.get("RUNPOD_POD_ID", "")
if pod:
    print("  https://%s-%d.proxy.runpod.net/\n" % (pod, PORT))
    print("If that 502s, port %d is not in this pod's HTTP ports - either add it (a template" % PORT)
    print("change needs the pod recreated) or use the Cloudflare tunnel below.")
else:
    print("RUNPOD_POD_ID is not set - this is not a RunPod pod, or it is an older image.")
    print("Use the Cloudflare tunnel below, or an SSH tunnel:")
    print("  ssh -L %d:localhost:%d root@<pod-host> -p <ssh-port>   then http://localhost:%d/"
          % (PORT, PORT, PORT))


### Option B — Cloudflare quick tunnel  *(no pod configuration, no account)*

Downloads `cloudflared` and prints a `https://<random>.trycloudflare.com` URL that proxies to the
server, websockets included. Leave the tunnel running for as long as you want the URL; re-run the
cell to get a fresh one.


In [ ]:
import re
os.makedirs(".bin", exist_ok=True)
CF = os.path.abspath(".bin/cloudflared")
if not os.path.exists(CF):
    print("downloading cloudflared ...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF)
    os.chmod(CF, 0o755)

sh("pkill -f 'cloudflared tunnel'; sleep 1", check=False)
tun = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://localhost:%d" % PORT],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public, t0 = None, time.time()
while time.time() - t0 < 90:
    line = tun.stdout.readline()
    if not line:
        break
    m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
    if m:
        public = m.group(0)
        break
if public:
    print("\n" + "=" * 72)
    print("  OPEN THIS IN CHROME:   %s/" % public)
    print("=" * 72)
    print("\nClick 'Start conversation', allow the microphone, and speak.")
else:
    print("could not get a tunnel URL; last output:")
    print(tun.stdout.read()[-2000:] if tun.stdout else "")


## 10 · Headless end-to-end check (no browser, no microphone)

Streams 20 s of recorded speech into the server, then silence, and reports what came back.
`PASS` = it heard the question, answered with speech and streamed video above 15 fps. The median
step time must stay under **80 ms** — that is the whole budget for one 80 ms audio frame. The
answer differs every run.


In [ ]:
sh("$(cat .python_path) tools/test_session.py %d 40" % PORT, check=False)


## 11 · Day-to-day: logs, restart, stop


In [ ]:
sh("tail -60 run.log", check=False)


In [ ]:
# step times and anything worth knowing about
sh("grep -aE 'step time|over budget|WARNING|Error|Traceback' run.log | tail -40", check=False)


In [ ]:
# restart under the watchdog instead (auto-restarts on a crash; logs to run.log + crashlogs/)
sh("pkill -f imtalker_try19_student_live.py; sleep 2; "
   "PORT=%d nohup setsid ./run_watchdog_x86.sh > /dev/null 2>&1 < /dev/null & "
   "sleep 3; pgrep -af run_watchdog_x86.sh" % PORT, check=False)


In [ ]:
# stop everything
sh("pkill -f run_watchdog_x86.sh; pkill -f imtalker_try19_student_live.py; "
   "pkill -f 'cloudflared tunnel'; sleep 2; "
   "nvidia-smi --query-gpu=memory.used --format=csv,noheader", check=False)


## 12 · Tuning and troubleshooting

### Knobs — set them in cell 8 (or export them and re-run `./run_x86.sh`)

| variable | default | effect |
|---|---|---|
| `PORT` | 8991 | server port (8998 is refused by the server) |
| `VOICE` | `Robert_5.pt` | voice prompt from `weights/voices/`. The depformer and the decoder were trained for Robert_5 — other voices sound clearly worse |
| `TEXT_PROMPT` | Robert, RB Labs | persona / system prompt |
| `PREBUFFER` | 3 | pre-roll chunks of 80 ms. Raise it if audio stutters on a slow network |
| `TRY19_INT4_G` | from the probe | `32` = int4 trunk (~14 GB VRAM), `0` = bf16 trunk (~26 GB, slower) |
| `TRY19_MIMI_GRAPH` | 1 | Mimi encoder CUDA graph. Set `0` if graph capture misbehaves on your card |
| `TRY19_RENDER_FP16` | 1 | fp16 renderer. `0` for fp32 (slower, 64 dB difference) |
| `GAIN` | 1.2 | motion gain of the avatar |

Example: `PORT=8080 TRY19_INT4_G=0 ./run_x86.sh 2>&1 | tee run.log`

### Troubleshooting

| symptom | fix |
|---|---|
| microphone blocked, no "allow" prompt | the page must be on `https://` or `http://localhost`. Use the Cloudflare tunnel or an SSH tunnel |
| the proxy URL 502s | port `8991` is not in the pod's HTTP ports, or the server has not finished loading — check `/health` first |
| `int4 tinygemm : UNAVAILABLE` in the probe | normal on some architectures. The trunk falls back to bf16 and then needs ~26 GB of VRAM: use a bigger GPU, or accept the slower path |
| `CUDA out of memory` while loading | something else holds VRAM (`nvidia-smi`), or the card is too small for the plan the probe chose |
| step 2 of `prepare_x86.sh` fails | the token is wrong, or the HF account has not accepted the `nvidia/personaplex-7b-v1` licence |
| no TensorRT engine | see `weights/trtexec_build.log`. Not fatal — Mimi's PyTorch SEANet decoder is used instead, a few ms slower per step with identical audio |
| browser connects then disconnects at once | `sphn` must be the **pip** 0.2.x build. `src/pylibs` holds an aarch64 `sphn` and must stay off `PYTHONPATH` on x86 — `run_x86.sh` already keeps it off |
| choppy audio, step time over 80 ms | `env/bin/pip uninstall uvloop`; make sure nothing else is using the GPU; run one server per GPU |
| `ImportError: libGL.so.1` | re-run cell 3 |
| disk full mid-download | the volume is under ~60 GB. `du -sh hf weights env .cache` shows where it went |

### Known behaviour (same as on the Orin)

* With very long continuous speech the avatar video can fall a few seconds behind; normal
  back-and-forth conversation stays ~0.1–0.15 s behind.
* The model's context window is 4 minutes, so very long sessions drift after ~4–6 minutes.
  Reloading the page starts a fresh session.
* Everything lives under `/workspace/speech2face`. On a pod restart with the same volume, skip
  cells 6 and 7 and go straight to cell 8.
